In [2]:
import os
import json
import re
import datetime
from time import time, sleep
from uuid import uuid4
from openai import OpenAI
from rdflib import Graph

# Módulos locales e importaciones de configuración
from searchInGraph import buscar_frecuentes_por_opcion, inferir_valor_adecuado
from formatHelper import extraer_support_category, extraer_cliente, extraer_parametro_gen
import config

# Inicialización del grafo
graph = Graph()
graph.parse("incident_triplets_convertido.ttl", format=config.TTL_FORMAT)

mi_model = "mistral:latest"

# Funciones auxiliares
def open_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return infile.read()

def save_file(filepath, content):
    # os.makedirs(os.path.dirname(filepath), exist_ok=True) # revisa el directorio antes si es necesario
    with open(filepath, 'w', encoding='utf-8') as outfile:
        outfile.write(content)

def load_json(filepath):
    with open(filepath, 'r', encoding='utf-8') as infile:
        return json.load(infile)

def save_json(filepath, payload):
    with open(filepath, 'w', encoding='utf-8') as outfile:
        json.dump(payload, outfile, ensure_ascii=False, sort_keys=True, indent=2)

def timestamp_to_datetime(unix_time):
    return datetime.datetime.fromtimestamp(unix_time).strftime("%A, %B %d, %Y at %I:%M%p %Z")

# Inicialización del cliente de OpenAI para Ollama
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

def text_completion(prompt, engine=config.MI_MODELO):
    max_retry = 5
    retry = 0

    while True:
        try:
            response = client.chat.completions.create(
                messages=[
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ],
                model=engine,
            )
            
            text = response.choices[0].message.content
            text = re.sub(r'[\r\n]+', '\n', text)
            text = re.sub(r'[\t ]+', ' ', text)
            return text
            
        except Exception as oops:
            retry += 1
            if retry >= max_retry:
                return "Model error: %s" % oops
            print('Error communicating with model:', oops)
            sleep(config.RETRY_DELAY_SECONDS)

In [3]:
print("Grafo cargado correctamente")
print(f"Número de triples: {len(graph)}")

Grafo cargado correctamente
Número de triples: 7425543


In [14]:
# Variables y configuraciones iniciales de ejecución
convo_length = 2 # se puede cambiar

unique_conv_id = str(uuid4())
prev_conv = ""
filename = unique_conv_id + '_log.txt'
log_file_path = os.path.join(config.LOGS_DIR, filename)

# Asegurar que el directorio de logs existe
if not os.path.exists(config.LOGS_DIR):
    os.makedirs(config.LOGS_DIR)

save_file(log_file_path, prev_conv)

primera = True
buscar = False
mi_opcion = None
cat_buscar = 0
graph_data = []
mis_datos = [None, None, None, None, None, None]

# Bucle principal de interacción
while True:

    if primera:
        a = input('\n\nUSER: ')
    
    primera = False
    buscar = True
    
    if a.lower() == "q":
        break
    
    opciones_si = {"y", "s", "yes", "si"}
    opciones_no = {"n", "no", "skip"}

    confirmado = False

    if graph_data:
        # Mostrar todas las opciones a la vez (estilo imagen)
        print(f"\n[Asistente] ¿Cuál es el valor para {config.DICCIONARIO_PREFIJOS[cat_buscar]}? (Responde con el número o 'si' si es la primera opción)")
        print("Opciones recomendadas (GraphRAG):")
        
        for i, opcion in enumerate(graph_data):
            print(f" {i+1}. {opcion}")
        
        print("(responde con número, s/si = #1, n/no = inferencia, q = salir)")
        #a = input('\nUSER: ').strip().lower()
        
        a = "1"
        
        
        
        
        if a == 'q':
            break 
        
        # CASO A: El usuario elige por número (1, 2, 3...)
        elif a.isdigit():
            idx = int(a) - 1
            if 0 <= idx < len(graph_data):
                mis_datos[cat_buscar] = graph_data[idx]
                confirmado = True
        
        # CASO B: El usuario dice "Sí" (se asume la opción #1 por defecto)
        elif a in opciones_si:
            mis_datos[cat_buscar] = graph_data[0]
            confirmado = True

        # CASO C: El usuario dice "No" explícitamente
        elif a in opciones_no:
            print("\nGraphRAG: Entendido. Buscando alternativas...")
            graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)
            buscar = False

    # Esto servirá si se quiere mantener un registro de las peticiones del usuario
    timestamp = time()
    timestring = timestamp_to_datetime(timestamp)
    message = '%s: %s - %s' % ('USER', timestring, a)
    
    '''
    0- Int_hasCustomer - Esta priori nos las dan
    1- hasUser - esto no existe? en teoría es hasUser pero no hay de eso, al menos en filtrado.ttl. Lo cambio a hasSupportCategory
    2- hasTypeInc
    3- incident_hasOrigin
    4- hasSupportGroup
    5- hasTechnician
    '''
    
    # TODO: Esto es algo burdo, mejorar mas tarde
    if mis_datos[0] is None:
        cliente = extraer_cliente(a)
        mis_datos[0] = cliente
        
    if mis_datos[1] is None:
        support_cat = extraer_support_category(a)
        mis_datos[1] = support_cat
    
    if buscar:
        if None not in mis_datos:
            print('\nGraphRAG: query acabada. La query es ' + str(mis_datos))
            break
            
        cat_buscar = mis_datos.index(None)
        graph_data = buscar_frecuentes_por_opcion(graph, mis_datos, cat_buscar)
        
        if not graph_data:
            graph_data = inferir_valor_adecuado(graph, mis_datos, cat_buscar)

        if graph_data:
            mi_opcion = graph_data[0]

    prev_conv = open_file(log_file_path)

    if not graph_data:
        data = "No se encontraron datos. Seguramente sea un error por parte del usuario. Pregunta si se ha introducido bien el grupo"
    else:
        data = "El campo a rellenar es " + config.DICCIONARIO_PREDICADOS[cat_buscar] + " y estas son las opciones\n" + str(mi_opcion)

    # --- ZONA DE INTEGRACIÓN LLM (Comentada) ---
    # prompt = open_file(config.CONTEXTO_FILE_PATH).replace('<<DATOS>>', data).replace('<<CONVERSACIÓN>>', prev_conv).replace('<<MENSAJE>>', a)
    # output = text_completion(prompt) #aquí se genera
    # timestamp = time()
    # timestring = timestamp_to_datetime(timestamp)
    # messageBot = '%s: %s - %s' % ('[Asistente]', timestring, output)
    # print('\n[Asistente]: %s' % output)
    
    # Guardado de los datos de la conversación
    # save_file(log_file_path, prev_conv + "\n" + message + "\n" + messageBot)



USER:  Hola quiero completar una query. Tengo el supportCategory_1497661091762302664 y la empresa ss



[Asistente] ¿Cuál es el valor para typeIncident? (Responde con el número o 'si' si es la primera opción)
Opciones recomendadas (GraphRAG):
 1. typeIncident__1
 2. typeIncident__2
(responde con número, s/si = #1, n/no = inferencia, q = salir)

[Asistente] ¿Cuál es el valor para incidentOrigin? (Responde con el número o 'si' si es la primera opción)
Opciones recomendadas (GraphRAG):
 1. incidentOrigin__2
 2. incidentOrigin__3
 3. incidentOrigin__1
 4. incidentOrigin__4
(responde con número, s/si = #1, n/no = inferencia, q = salir)

[Asistente] ¿Cuál es el valor para supportGroup? (Responde con el número o 'si' si es la primera opción)
Opciones recomendadas (GraphRAG):
 1. supportGroup_14976631762302662
 2. supportGroup_14976691762302662
 3. supportGroup_149761521762302662
 4. supportGroup_149761661762302662
(responde con número, s/si = #1, n/no = inferencia, q = salir)

[Asistente] ¿Cuál es el valor para employee? (Responde con el número o 'si' si es la primera opción)
Opciones recomend